## Method 1

In [1]:
import re
import numpy as np

from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


##### Define the Class Critique

In [2]:
@dataclass
class Critique:
    critique_id: str
    painting_id: str
    critic_name: str
    title: str
    text: str
    published_at: datetime



##### Helper Functions

In [3]:
def tokenize_simple(text: str) -> set:
    """
    Convert text into a set of lowercase word tokens.
    """
    return set(re.findall(r"[a-zA-Z']+", text.lower()))


def keyword_overlap(query: str, text: str) -> float:
    """
    Measure the fraction of query words that also appear in the critique.
    Bounds: 0 <= keyword_overlap <= 1
    """
    query_words = tokenize_simple(query)
    text_words = tokenize_simple(text)

    if not query_words:
        return 0.0

    overlap = query_words.intersection(text_words)
    return len(overlap) / len(query_words)


def length_norm(text: str, target_length: int = 250) -> float:
    """
    Reward critiques that contain a reasonable amount of detail.
    A critique with target_length words receives a score of 1.
    Longer critiques are capped at 1, so length is not rewarded indefinitely.
    Bounds:
        0 <= length_norm <= 1
    """
    word_count = len(tokenize_simple(text))

    return min(word_count / target_length, 1.0)


def recency_score(
    published_at: datetime,
    now: Optional[datetime] = None,
    half_life_days: int = 3650
) -> float:
    """
    Apply a gentle exponential recency decay.
    With the default half-life of 3650 days, or about 10 years:
        - a critique published today receives a score close to 1
        - a critique published 10 years ago receives a score close to 0.5
        - a critique published 20 years ago receives a score close to 0.25
    Bounds:
        0 < recency_score <= 1
    """
    if now is None:
        now = datetime.now()

    age_days = max((now - published_at).days, 0)

    return 0.5 ** (age_days / half_life_days)



##### Defining Class HeuristicCritiqueMatcher

In [4]:
class HeuristicCritiqueMatcher:
    """
    Given a new critique of a painting, retrieve previous critiques
    of the same painting that are most similar to it.
    This first-generation method uses transparent handcrafted features:
    1. TF-IDF cosine similarity
        2. keyword overlap
        3. length normalization
        4. recency
    The final score is a normalized heuristic relevance score in [0, 1].
    It is not a probability.
    """

    def __init__(self, critiques: List[Critique], painting_id: str):

        # Keep only critiques for the selected painting.
        self.critiques = [
            c for c in critiques
            if c.painting_id == painting_id
        ]

        if not self.critiques:
            raise ValueError(f"No critiques found for painting_id={painting_id}")

        # Searchable text combines critique title and critique body.
        self.texts = [
            f"{c.title} {c.text}"
            for c in self.critiques
        ]

        # TF-IDF converts each critique into a sparse numerical vector.
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            lowercase=True,
            min_df=1
        )

        # Document-term matrix:
        # rows    = critiques
        # columns = terms, including unigrams and bigrams
        # values  = TF-IDF weights
        self.matrix = self.vectorizer.fit_transform(self.texts)

        # Fixed handcrafted weights.
        # TF-IDF similarity receives the largest weight because it is
        # the main text-similarity signal in this method.
        self.weights = {
            "tfidf_sim": 1.20,
            "keyword_overlap": 0.60,
            "length_norm": 0.20,
            "recency": 0.15,
        }

        # Used to normalize the final weighted score to [0, 1].
        self.max_possible_score = sum(self.weights.values())

    def _score_one(
        self,
        query: str,
        critique: Critique,
        tfidf_sim: float,
        now: Optional[datetime] = None
    ) -> float:
        """
        Score one previous critique against the new query critique.
        Each feature is designed to lie in [0, 1].
        We still apply clipping as a safety check.
        The raw weighted score is divided by the sum of the weights,
        producing a final normalized score in [0, 1].
        """

        full_text = f"{critique.title} {critique.text}"

        features = {
            "tfidf_sim": np.clip(tfidf_sim, 0.0, 1.0),

            "keyword_overlap": np.clip(
                keyword_overlap(query, full_text),
                0.0,
                1.0
            ),

            "length_norm": np.clip(
                length_norm(critique.text),
                0.0,
                1.0
            ),

            "recency": np.clip(
                recency_score(critique.published_at, now=now),
                0.0,
                1.0
            ),
        }

        raw_score = sum(
            self.weights[name] * features[name]
            for name in self.weights
        )

        normalized_score = raw_score / self.max_possible_score

        return float(normalized_score)

    def score_query(
        self,
        query: str,
        top_k: int = 5,
        now: Optional[datetime] = None
    ) -> List[Tuple[Critique, float]]:
        """
        Rank previous critiques of the same painting by similarity
        to the new critique.
        Parameters
        ----------
        query:
            The new student critique or query text.
        top_k:
            Number of highest-scoring critiques to return.
         now:
            Optional reference date for computing recency.
       Returns
        -------
        A list of (Critique, score) pairs, sorted from highest score to lowest.
        """

        # Convert the new critique into the same TF-IDF vector space.
        q_vec = self.vectorizer.transform([query])

        # Compute TF-IDF cosine similarity against all previous critiques.
        similarities = cosine_similarity(self.matrix, q_vec).ravel()

        # Compute the final normalized heuristic score.
        scores = [
            self._score_one(
                query=query,
                critique=self.critiques[i],
                tfidf_sim=float(similarities[i]),
                now=now
            )
            for i in range(len(self.critiques))
        ]

        # Sort scores in descending order.
        top_indices = np.argsort(-np.array(scores))[:top_k]

        return [
            (self.critiques[i], float(scores[i]))
            for i in top_indices
        ]

##### Matching a New Student Critique

In [5]:
critiques = [
    Critique(
        critique_id="c001",
        painting_id="painting_001",
        critic_name="Expert A",
        title="Light and Stillness",
        text="""
        The painting uses soft light and a restrained color palette
        to create a mood of quiet reflection. The central figure appears
        isolated but dignified, while the balanced composition gives the
        scene a sense of stillness and emotional control.
        """,
        published_at=datetime(2021, 5, 10)
    ),

    Critique(
        critique_id="c002",
        painting_id="painting_001",
        critic_name="Expert B",
        title="Color and Surface",
        text="""
        The work is most striking in its handling of pigment and surface.
        The brushwork is layered and tactile, with subtle tonal transitions
        that reveal the artist's technical control over color and texture.
        """,
        published_at=datetime(2018, 9, 3)
    ),

    Critique(
        critique_id="c003",
        painting_id="painting_001",
        critic_name="Expert C",
        title="Historical Symbolism",
        text="""
        The painting should be read through its historical context.
        Its gestures, objects, and spatial arrangement suggest a broader
        symbolic program connected to religious and cultural traditions.
        """,
        published_at=datetime(2005, 1, 20)
    ),

    Critique(
        critique_id="c004",
        painting_id="painting_001",
        critic_name="Expert D",
        title="Psychological Interior",
        text="""
        The emotional power of the painting comes from its psychological
        restraint. The figure seems inward-looking, almost withdrawn,
        and the muted background intensifies the feeling of solitude,
        introspection, and quiet human vulnerability.
        """,
        published_at=datetime(2023, 11, 15)
    ),

    Critique(
        critique_id="c005",
        painting_id="painting_001",
        critic_name="Expert E",
        title="Narrative and Gesture",
        text="""
        The narrative force of the painting lies in the figure's gesture.
        The position of the hands, the direction of the gaze, and the
        surrounding objects suggest an unfolding story rather than a
        purely formal arrangement.
        """,
        published_at=datetime(2012, 6, 8)
    ),
]

student_critique_text = """
The painting creates a quiet emotional atmosphere, yet very powerful.
The soft light and restrained color palette
make the central figure feel isolated yet dignified. The background
does not compete with the subject; instead, it deepens the mood of
reflection and stillness. Overall, the work feels intimate,
psychological, and carefully composed.
"""
matcher = HeuristicCritiqueMatcher(
    critiques=critiques,
    painting_id="painting_001"
)

results = matcher.score_query(
    query=student_critique_text,
    top_k=5
)

for critique, score in results:
    print(f"{critique.title} | {critique.critic_name} | score = {score:.3f}")

Light and Stillness | Expert A | score = 0.531
Psychological Interior | Expert D | score = 0.296
Narrative and Gesture | Expert E | score = 0.224
Color and Surface | Expert B | score = 0.212
Historical Symbolism | Expert C | score = 0.096
